# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itz-me-sree/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
!pip -q install duckdb huggingface_hub

In [17]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

In [18]:
import pandas as pd

import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [19]:
df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    'hf://datasets/Flyrank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 6)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My baseline rule prioritizes pages with existing search demand and an opportunity to improve clicks. I use search impressions as the volume signal and CTR together with average position as the performance signal.


Signal 1 — Search volume
Verdict: CONFIRMED.

Impressions provide a useful measure of existing search demand, with a large range from low-volume pages to pages receiving 100+ impressions.

Signal 2 — CTR vs position
Verdict: CONFIRMED.

The share of pages receiving clicks decreases as average position gets worse: 17.98% for positions 1–5, 12.54% for 5–10, 10.31% for 10–20, and 5.52% for 20+.

In [21]:
# ML-07 — Signal checks

# Calculate CTR safely
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, pd.NA)
)

# -----------------------------
# Signal 1: Search volume
# -----------------------------

print("SIGNAL 1 — Search volume (GSC impressions)")

volume_buckets = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 10, 50, 100, float("inf")],
    labels=["0-10", "11-50", "51-100", "100+"]
)

volume_table = (
    df.groupby(volume_buckets, observed=False)
      .size()
      .reset_index(name="n")
)

display(volume_table)


# -----------------------------
# Signal 2: CTR vs position
# -----------------------------

print("SIGNAL 2 — CTR vs average position")

position_buckets = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 5, 10, 20, float("inf")],
    labels=["1-5", "5-10", "10-20", "20+"]
)

ctr_position_table = (
    df.assign(position_bucket=position_buckets)
      .groupby("position_bucket", observed=False)
      .agg(
          n=("content_hash_id", "size"),
          pages_with_clicks=("gsc_clicks", lambda x: (x > 0).sum()),
          mean_ctr=("ctr", "mean"),
          median_impressions=("gsc_impressions", "median")
      )
      .reset_index()
)

ctr_position_table["click_rate"] = (
    ctr_position_table["pages_with_clicks"] /
    ctr_position_table["n"]
)

display(ctr_position_table)

SIGNAL 1 — Search volume (GSC impressions)


,gsc_impressions,n
0,0-10,7761951
1,11-50,1054396
2,51-100,391548
3,100+,633483


SIGNAL 2 — CTR vs average position


,position_bucket,n,pages_with_clicks,mean_ctr,median_impressions,click_rate
0,1-5,1099936,197732,0.00454,28.0,0.179767
1,5-10,920359,115435,0.003083,19.0,0.125424
2,10-20,519223,53551,0.00277,20.0,0.103137
3,20+,908354,50112,0.001289,9.0,0.055168


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
# ML-07 — Build baseline ranked queue
# DuckDB version: avoids copying the full 8M-row dataset in pandas

import os

os.makedirs("work/outputs", exist_ok=True)

con.sql("""
COPY (
    WITH base AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,

            CASE
                WHEN gsc_impressions > 0
                THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
                ELSE NULL
            END AS ctr

        FROM read_parquet(
            'hf://datasets/Flyrank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )
    ),

    scored AS (
        SELECT
            *,

            -- Higher search demand = higher score
            CASE
                WHEN gsc_impressions > 100 THEN 3
                WHEN gsc_impressions > 50 THEN 2
                WHEN gsc_impressions > 10 THEN 1
                ELSE 0
            END AS volume_score,

            -- Better position gives more useful CTR context
            CASE
                WHEN gsc_avg_position IS NULL THEN 0
                WHEN gsc_avg_position <= 5 THEN 3
                WHEN gsc_avg_position <= 10 THEN 2
                WHEN gsc_avg_position <= 20 THEN 1
                ELSE 0
            END AS position_score,

            -- Lower CTR = larger possible CTR opportunity
            CASE
                WHEN ctr IS NULL THEN 3
                WHEN ctr <= 0.01 THEN 3
                WHEN ctr <= 0.03 THEN 2
                WHEN ctr <= 0.10 THEN 1
                ELSE 0
            END AS ctr_score

        FROM base
    ),

    final_scores AS (
        SELECT
            *,

            volume_score + position_score + ctr_score AS score,

            CASE
                WHEN volume_score >= 2 AND ctr_score >= 2
                    THEN 'HIGH_VOLUME_LOW_CTR'
                WHEN volume_score >= 2
                    THEN 'HIGH_VOLUME'
                WHEN ctr_score >= 2
                    THEN 'LOW_CTR_OPPORTUNITY'
                ELSE 'LOW_PRIORITY'
            END AS reason_code,

            CASE
                WHEN volume_score + position_score + ctr_score >= 6
                    THEN 'REVIEW_CTR'
                WHEN volume_score + position_score + ctr_score >= 4
                    THEN 'REVIEW'
                ELSE 'MONITOR'
            END AS action

        FROM scored
    )

    SELECT
        ROW_NUMBER() OVER (
            ORDER BY score DESC, gsc_impressions DESC
        ) AS rank,
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ctr,
        score,
        reason_code,
        action

    FROM final_scores
    ORDER BY score DESC, gsc_impressions DESC
)
TO 'work/outputs/baseline_action_score.csv'
(HEADER, DELIMITER ',');
""")

print("CSV created successfully.")
print("work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV created successfully.
work/outputs/baseline_action_score.csv


In [24]:
top10 = con.sql("""
SELECT *
FROM read_csv_auto('work/outputs/baseline_action_score.csv')
ORDER BY rank
LIMIT 10
""").df()

display(top10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
0,1,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.083350,0.000025,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1,2,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,2.197507,0.006411,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
2,3,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,2.764916,0.000051,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
3,4,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,2.195988,0.007051,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
4,5,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,2.188397,0.006355,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
5,6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,2.181348,0.006405,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
6,7,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,2.242501,0.006791,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
7,8,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,2.309046,0.006404,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
8,9,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,0.000000,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
9,10,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,0.000000,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Top-10 review**

For each of the top ten rows, I record the action, reason code, confidence, and what evidence would make the recommendation wrong.


1. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 40,084 impressions but only 1 click gives an extremely low CTR. This would be wrong if the high impression count came from an unusual tracking or search-query mix rather than a real content opportunity.

2. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 39,305 impressions and 252 clicks produce a low CTR despite an average position of about 2.20. This would be wrong if the page is already performing appropriately for its search intent or if SERP features explain the click pattern.

3. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 39,003 impressions with only 2 clicks is a very strong low-CTR signal. This would be wrong if the impressions are caused by irrelevant queries or a measurement anomaly.

4. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 38,436 impressions and 271 clicks with an average position near 2.20 indicate substantial search demand with room to investigate CTR. This would be wrong if the low CTR is expected for the page's query intent or SERP layout.

5. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 35,404 impressions and 225 clicks indicate high demand and relatively low CTR. This would be wrong if the page is intentionally targeting queries where clicks are naturally uncommon.

6. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 34,817 impressions and 223 clicks with a position near 2.18 make this a strong review candidate. This would be wrong if the observed CTR is normal for the page's actual search queries.

7. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 34,606 impressions and 235 clicks show strong demand with low CTR. This would be wrong if the traffic pattern is temporary or caused by unusual queries.

8. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 33,571 impressions and 215 clicks with an average position around 2.30 make CTR worth reviewing. This would be wrong if SERP features or query intent explain the limited clicks.

9. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 33,383 impressions but zero clicks is a strong signal for investigation. This would be wrong if impressions are caused by irrelevant queries, reporting issues, or a page/query mismatch that cannot be fixed through content changes.

10. REVIEW_CTR — HIGH_VOLUME_LOW_CTR — High confidence. 32,958 impressions and zero clicks indicate a large apparent opportunity. This would be wrong if the impressions are not meaningful search demand or if the zero clicks result from a measurement problem.
---



Review note: Several top-ranked rows represent the same content item on different report dates. This is a limitation of this row-level baseline: it can repeatedly prioritize the same content across days. A future version could aggregate performance by content item before creating the action queue.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Weak picks + leakage check

The weakest picks are the rows with very high impressions but very low or zero clicks. These are useful investigation candidates, but the rule cannot prove that a content change will improve performance. A high impression count can also come from broad or irrelevant queries, SERP features, or measurement issues.

The baseline uses only March 2026 performance fields: impressions, clicks, and average position. CTR is calculated from clicks and impressions. It does not use the proposed `needs_refresh` label, future dates, product flags, or future-window information.

One limitation is that the queue can contain the same content item on multiple report dates because the source is daily performance data. This can cause repeated recommendations for the same content item.

In [27]:
# ML-07 — Weak picks + leakage check

print("=== LEAKAGE CHECK ===")

# Columns actually used by the baseline
baseline_inputs = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("Inputs used by baseline:")
print(baseline_inputs)

# Confirm the proposed target is NOT used
print("\nTarget leakage check:")
print("needs_refresh used in scoring: NO")

# Confirm the scoring window
print("\nTime-window check:")
print("Baseline source window: March 2026")
print("Future-window data used: NO")

# Check repeated content IDs in Top 10
print("\n=== TOP-10 WEAK-PICK CHECK ===")

top10_content_counts = (
    top10.groupby("content_hash_id")
         .size()
         .sort_values(ascending=False)
)

print("Repeated content items in Top 10:")
print(top10_content_counts[top10_content_counts > 1])

# Show the weakest CTR candidates
print("\nTop-10 rows with lowest CTR:")
display(
    top10[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr",
            "score",
            "reason_code",
            "action"
        ]
    ].sort_values("ctr").head(10)
)

print("\nLeakage check complete.")

=== LEAKAGE CHECK ===
Inputs used by baseline:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Target leakage check:
needs_refresh used in scoring: NO

Time-window check:
Baseline source window: March 2026
Future-window data used: NO

=== TOP-10 WEAK-PICK CHECK ===
Repeated content items in Top 10:
content_hash_id
content_eadb33b5df496f4a    6
content_44f34c0a90047651    2
dtype: int64

Top-10 rows with lowest CTR:


,rank,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
9,10,content_44f34c0a90047651,32958,0,0.132532,0.000000,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
8,9,content_fec55986a1868d62,33383,0,0.181500,0.000000,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
0,1,content_44f34c0a90047651,40084,1,0.083350,0.000025,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
2,3,content_34a70fea29d15f24,39003,2,2.764916,0.000051,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
4,5,content_eadb33b5df496f4a,35404,225,2.188397,0.006355,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
7,8,content_eadb33b5df496f4a,33571,215,2.309046,0.006404,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
5,6,content_eadb33b5df496f4a,34817,223,2.181348,0.006405,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
1,2,content_eadb33b5df496f4a,39305,252,2.197507,0.006411,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
6,7,content_eadb33b5df496f4a,34606,235,2.242501,0.006791,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR
3,4,content_eadb33b5df496f4a,38436,271,2.195988,0.007051,9,HIGH_VOLUME_LOW_CTR,REVIEW_CTR



Leakage check complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.